# Spatial deconvolution with RCTD

**RCTD** (Robust Cell Type Decomposition, [Cable et al., *Nat. Biotechnol.* 2022](https://doi.org/10.1038/s41587-021-00830-w)) is the deconvolution method 10x Genomics officially recommends for Visium / Visium HD spot decomposition. The original implementation (`spacexr`) is in R; this notebook uses the pure-Python port [`rctd-py`](https://github.com/p-gueguen/rctd-py).

RCTD is wired through the same `ov.space.Deconvolution` class that already wraps Tangram and cell2location, so the API is identical to the other deconvolution backends — the only thing you change is `method='RCTD'`.

We exercise it on the **same Lymph Node dataset** used by the canonical `t_decov` tutorial:

- `ov.datasets.sc_ref_Lymph_Node()` — Lymph Node scRNA-seq reference (annotated by `Subset`)
- `sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")` — 10x Visium V1 Human Lymph Node

so the cell-type proportions can be compared directly against Tangram / cell2location runs in the existing tutorial.

## Environment setup

In [ ]:
import omicverse as ov
ov.style(font_path='Arial')

import scanpy as sc
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

## Step 1: Prepare the scRNA-seq reference

RCTD trains a per-cell-type expression profile from the scRNA-seq reference and uses it to fit each spatial pixel. The reference must contain **raw counts** in either `adata.X` or `adata.layers['counts']` — RCTD's likelihood model is defined over count data, not log-normalised values.

`ov.datasets.sc_ref_Lymph_Node()` ships counts in `layers['counts']`, so the integration picks them up automatically (and falls back to `.X` when the layer isn't present).

In [ ]:
adata_sc = ov.datasets.sc_ref_Lymph_Node()
print(adata_sc)
print()
print("cell types:", adata_sc.obs['Subset'].nunique())
print("counts layer present:", 'counts' in adata_sc.layers)

## Step 2: Prepare the spatial transcriptomics dataset

We use the public Visium V1 Human Lymph Node sample. `sc.datasets.visium_sge` returns raw counts directly in `.X`, which is what RCTD wants.

In [ ]:
adata_sp = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata_sp.obs['sample'] = list(adata_sp.uns['spatial'].keys())[0]
adata_sp.var_names_make_unique()
# Keep a copy of the raw counts in `layers['counts']` for downstream
# tools that expect that convention (and so we can normalize `.X` later
# for plotting without losing the input RCTD will train against).
if 'counts' not in adata_sp.layers:
    adata_sp.layers['counts'] = adata_sp.X.copy()
print(adata_sp)

## Step 3: Run RCTD

Construct the `Deconvolution` object exactly like for Tangram / cell2location and pass `method='RCTD'`.

**Important note on preprocessing:** unlike the Tangram / cell2location flow, **we do not call `preprocess_sc` / `preprocess_sp`** before RCTD. Those helpers replace `.X` with log-normalised values and subset to HVGs / SVGs — but RCTD's likelihood model wants raw counts and selects its own informative gene set internally (the `gene_cutoff` / `fc_cutoff` controls in `RCTDConfig`).

The integration pulls raw counts from `layers['counts']` if present, otherwise from `.X`.

In [ ]:
decov_obj = ov.space.Deconvolution(
    adata_sc=adata_sc,
    adata_sp=adata_sp,
)

In [ ]:
decov_obj.deconvolution(
    method='RCTD',
    celltype_key_sc='Subset',
    rctd_kwargs={
        # Mode controls how RCTD reports per-pixel composition:
        # - 'full'    : per-pixel weights over *all* cell types (pixels × types proportion matrix)
        # - 'doublet' : at most 2 cell types per pixel (RCTD paper default for spot-level Visium)
        # - 'multi'   : 1–N cell types per pixel, N from `RCTDConfig.MAX_MULTI_TYPES`
        'mode': 'full',
        'cell_min': 25,         # drop reference cell types with < cell_min cells
        'n_max_cells': 10000,   # subsample the reference for memory / runtime
        'min_UMI': 100,
        # Forward to RCTDConfig — see rctd-py docs for the full list.
        'config': {
            'UMI_min': 100,
            'UMI_max': 20_000_000,
        },
        'batch_size': 'auto',
    },
)

## Step 4: Inspect the result

`decov_obj.adata_cell2location` is the canonical omicverse output — a `(kept_pixels × n_cell_types)` AnnData of row-normalised proportions, with the spatial coordinates / `obsm` carried over from `adata_sp`. Same shape Tangram and cell2location populate, so all the downstream plotting in the existing `t_decov` tutorial works unchanged.

Two extras that are RCTD-specific:

- `decov_obj.rctd_result` — the raw `FullResult` / `DoubletResult` object from `rctd-py`, with the unnormalised `weights`, the `pixel_mask` (which pixels passed the UMI filter), and `converged` flags per pixel.
- `decov_obj.adata_sp.obsm['rctd_proportions']` — the same proportion matrix re-indexed back onto every pixel of the original `adata_sp` (filtered pixels get all-zero rows). Convenient for `ov.pl.spatial(color=<cell_type>)`.

In [ ]:
decov_obj.adata_cell2location

In [ ]:
import numpy as np
props = np.asarray(decov_obj.adata_cell2location.X)
print("row-sums (proportion check) min/max:", props.sum(axis=1).min(), props.sum(axis=1).max())
print("top cell types by mean proportion:")
ct_mean = decov_obj.adata_cell2location.X.mean(axis=0)
import pandas as pd
print(pd.Series(ct_mean, index=decov_obj.adata_cell2location.var_names)
        .sort_values(ascending=False).head(10))

## Step 5: Spatial visualisation

Re-using the same panel of canonical lymph-node markers as the `t_decov` tutorial so the RCTD output is directly comparable to the Tangram / cell2location results in that notebook.

In [ ]:
annotation_list = ['B_Cycling', 'B_GC_LZ', 'T_CD4+_TfH_GC', 'FDC',
                   'B_naive', 'T_CD4+_naive', 'B_plasma', 'Endo']
available = [a for a in annotation_list if a in decov_obj.adata_cell2location.var_names]
print('Plotting:', available)

sc.pl.spatial(
    decov_obj.adata_cell2location,
    cmap='magma',
    color=available,
    ncols=4,
    img_key='hires',
    vmin=0, vmax='p99.2',
)

### 5.2 Multi-cell-type overlay

RCTD's per-pixel proportion matrix can be drawn as an overlapping multi-channel image — one cell type per channel — to read out fine-grained tissue architecture (e.g. germinal-centre vs. naive-B regions in the lymph node).

In [ ]:
color_dict = dict(zip(adata_sc.obs['Subset'].cat.categories,
                       adata_sc.uns['Subset_colors']))
clust_labels = available[:5]
clust_col = ['' + str(i) for i in clust_labels]

In [ ]:
import matplotlib as mpl
with mpl.rc_context({'figure.figsize': (6, 6), 'axes.grid': False}):
    fig = ov.pl.plot_spatial(
        adata=decov_obj.adata_cell2location,
        color=clust_col,
        labels=clust_labels,
        show_img=True,
        style='fast',
        max_color_quantile=0.992,
        circle_diameter=4,
        colorbar_position='right',
    )


### 5.3 Region-of-interest pie plot

Crop a 1 mm sub-region first — at full-slide scale 4 000+ pies overlap and become unreadable. We re-use `ov.space.crop_space_visium`, the same helper the existing `t_decov` tutorial uses, so the panel is directly comparable.

In [ ]:
adata_s = ov.space.crop_space_visium(
    decov_obj.adata_cell2location,
    crop_loc=(0, 0),
    crop_area=(500, 1000),
    library_id=list(decov_obj.adata_cell2location.uns['spatial'].keys())[0],
    scale=1,
)
print(f'Subset: {adata_s.n_obs} pixels')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax = ov.pl.add_pie2spatial(
    adata_s,
    cell_type_columns=available[:5],
    ax=ax,
    pie_radius=12,
    colors=[color_dict[c] for c in available[:5]],
)
ax.set_title('RCTD per-pixel cell-type composition (zoomed)')
plt.show()


## Summary

We added RCTD as a new backend to `ov.space.Deconvolution`. The integration:

- accepts the same `(adata_sp, adata_sc)` constructor as Tangram / cell2location,
- automatically pulls raw counts from `adata.layers['counts']` (or falls back to `.X`),
- forwards every RCTD knob through `rctd_kwargs` (mode / cell_min / batch_size / sigma_override / a `config` dict for `RCTDConfig`),
- reshapes the `FullResult` / `DoubletResult` / `MultiResult` into the canonical `adata_cell2location` schema (pixels × cell-types proportion matrix, row-sums = 1),
- preserves the spatial `obsm` so all the existing visualisation helpers (`sc.pl.spatial`, `ov.space.crop_space_visium`, `ov.pl.add_pie2spatial`) work unchanged.

Compared with the other backends already in the tutorial:

| Method | Speed (this dataset) | GPU? | Notes |
|---|---|---|---|
| Tangram | 15–30 min | yes | Mapping-based; needs HVG preprocessing |
| cell2location | 30–120 min | recommended | Bayesian; large `max_epochs`; needs HVG preprocessing |
| **RCTD** | **5–15 min** | yes (`device` in `RCTDConfig`) | **No SVG/HVG step**; uses raw counts directly |